In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import Voronoi
from shapely.geometry import MultiPoint, Point, Polygon


def generar_voronoi_urbano(puntos_df, limite_mapa=None):
    """Genera polígonos de Voronoi recortados al área del mapa."""
    coords = puntos_df[["longitud", "latitud"]].values
    vor = Voronoi(coords)

    lines = []
    # Convertir las crestas/líneas de Voronoi a geometrías
    for idx, reg_num in enumerate(vor.point_region):
        indices = vor.regions[reg_num]
        if -1 not in indices and len(indices) > 0:
            poligono = Polygon([vor.vertices[i] for i in indices])
            lines.append(poligono)

    # Crear GeoDataFrame con los polígonos
    gdf_voronoi = gpd.GeoDataFrame(geometry=lines, crs="EPSG:4326")
    return gdf_voronoi


# 1. Definir los puntos del mapa (Ejemplo: Coordenadas de la ciudad)
data_colegios = {
    "nombre": ["Colegio A", "Colegio B", "Colegio C", "Colegio D"],
    "latitud": [4.6097, 4.6150, 4.6200, 4.6050],
    "longitud": [-74.0817, -74.0700, -74.0850, -74.0750],
}
df_colegios = pd.DataFrame(data_colegios)

# 2. Generar Voronoi
gdf_colegios = gpd.GeoDataFrame(
    df_colegios,
    geometry=gpd.points_from_xy(df_colegios.longitud, df_colegios.latitud),
    crs="EPSG:4326",
)
gdf_vor = generar_voronoi_urbano(df_colegios)

# 3. Visualizar en un gráfico
fig, ax = plt.subplots(figsize=(10, 8))
gdf_vor.plot(
    ax=ax, facecolor="none", edgecolor="blue", linewidth=1.5, label="Fronteras"
)
gdf_colegios.plot(ax=ax, color="red", markersize=50, label="Colegios")
plt.title("Diagrama de Voronoi - Cobertura de Colegios")
plt.legend()
plt.show()